# 🔧 Actividad 1: Configuración del Proyecto Python
---
**Tesis:** Predicción de Producción de Limón mediante LSTM Multimodal  
**Pipeline:** Fase 1 — Ingeniería de Datos  
**Objetivo:** Configurar el entorno de trabajo, librerías y estructura de carpetas para garantizar la reproducibilidad total del pipeline.

### Fuentes de Datos del Proyecto
| Fuente | Tipo | Archivo |
|:-------|:-----|:--------|
| MIDAGRI (SISAGRI) | Producción agrícola | `Sisagri_2016_2025.xlsx` |
| INDECI (SINPAD) | Emergencias y peligros | `resumen_emergencias_2003_2024.xlsx`, DBFs 2021-2023 |
| Agraria.pe | Noticias agrícolas | `agro_news_2021.csv` a `agro_news_2025.csv` |
| NASA POWER | Variables climáticas | `nasa_climatic_raw_values.csv`, `nasa_long_raw.csv` |


In [1]:
# ==========================================================================
# 1.1 Importación de Librerías
# ==========================================================================
import os
import sys
import shutil
import glob
import warnings

# Data Science
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Base de Datos
from sqlalchemy import create_engine, text

# NLP & ML
from sklearn.preprocessing import StandardScaler

# Lectura de archivos DBF (INDECI SINPAD)
from dbfread import DBF

# Serialización
import joblib

# Configuración global
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 200)
sns.set_theme(style='whitegrid', palette='muted')

# Navegar a la raíz del proyecto
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir(os.path.abspath('..'))
print(f"Directorio de trabajo: {os.getcwd()}")

print("✅ Todas las librerías cargadas correctamente.")


Directorio de trabajo: C:\Machine-learming\Machine-Learning-Multimodal--Agro-NLP-Clima-
✅ Todas las librerías cargadas correctamente.


In [2]:
# ==========================================================================
# 1.2 Creación de la Estructura de Carpetas
# ==========================================================================
# Definición de la arquitectura del proyecto (convención Data Engineering)
DIRS = {
    'raw':       os.path.join('data', '01_raw'),
    'raw_midagri': os.path.join('data', '01_raw', 'midagri'),
    'raw_indeci':  os.path.join('data', '01_raw', 'indeci'),
    'raw_news':    os.path.join('data', '01_raw', 'agraria_pe'),
    'interim':   os.path.join('data', '02_interim'),
    'processed': os.path.join('data', '03_processed'),
    'reports':   os.path.join('data', '04_reports'),
    'database':  'database',
    'scalers':   os.path.join('models', 'scalers'),
    'raw_nasa':       os.path.join('data', '01_raw', 'nasapower'),
    'interim_nasa':   os.path.join('data', '02_interim_nasa'),
    'processed_nasa': os.path.join('data', '03_processed_nasa'),
}

for key, path in DIRS.items():
    os.makedirs(path, exist_ok=True)
    print(f"  📁 {path}")

print("\n✅ Estructura de carpetas creada exitosamente (incluyendo NASA).")


  📁 data\01_raw
  📁 data\01_raw\midagri
  📁 data\01_raw\indeci
  📁 data\01_raw\agraria_pe
  📁 data\02_interim
  📁 data\03_processed
  📁 data\04_reports
  📁 database
  📁 models\scalers
  📁 data\01_raw\nasapower
  📁 data\02_interim_nasa
  📁 data\03_processed_nasa

✅ Estructura de carpetas creada exitosamente (incluyendo NASA).


In [3]:
# ==========================================================================
# 1.3 Copia de Archivos Fuente a 01_raw/
# ==========================================================================
# Copiar desde la estructura anterior a la nueva, manteniendo trazabilidad.

# --- MIDAGRI ---
src_midagri = os.path.join('data', 'raw', 'midagri', 'Sisagri_2016_2025.xlsx')
dst_midagri = os.path.join(DIRS['raw_midagri'], 'Sisagri_2016_2025.xlsx')
if os.path.exists(src_midagri) and not os.path.exists(dst_midagri):
    shutil.copy2(src_midagri, dst_midagri)
    print(f"  ✅ MIDAGRI copiado: {dst_midagri}")
elif os.path.exists(dst_midagri):
    print(f"  ℹ️  MIDAGRI ya existe en destino.")
else:
    print(f"  ⚠️  MIDAGRI no encontrado en: {src_midagri}")

# --- INDECI ---
indeci_files = [
    'resumen_emergencias_2003_2024.xlsx',
    'resumen_peligros_2003_2024.xlsx',
    'piura_emergencias.xlsx',
    'piura_peligros.xlsx',
]
for fname in indeci_files:
    src = os.path.join('data', 'raw', 'indeci', fname)
    dst = os.path.join(DIRS['raw_indeci'], fname)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst)
        print(f"  ✅ INDECI copiado: {fname}")
    elif os.path.exists(dst):
        print(f"  ℹ️  INDECI ya existe: {fname}")

# Copiar carpetas DBF (shapefiles extraídos)
for year_folder in ['E_2021', 'E_2022', 'E_2023']:
    src_dir = os.path.join('data', 'raw', 'indeci', year_folder)
    dst_dir = os.path.join(DIRS['raw_indeci'], year_folder)
    if os.path.exists(src_dir) and not os.path.exists(dst_dir):
        shutil.copytree(src_dir, dst_dir)
        print(f"  ✅ INDECI DBF copiado: {year_folder}/")
    elif os.path.exists(dst_dir):
        print(f"  ℹ️  INDECI DBF ya existe: {year_folder}/")

# --- AGRARIA.PE ---
for csv_file in glob.glob(os.path.join('data', 'raw', 'agraria_pe', 'agro_news_*.csv')):
    fname = os.path.basename(csv_file)
    dst = os.path.join(DIRS['raw_news'], fname)
    if not os.path.exists(dst):
        shutil.copy2(csv_file, dst)
        print(f"  ✅ AGRARIA copiado: {fname}")
    else:
        print(f"  ℹ️  AGRARIA ya existe: {fname}")

# --- NASA POWER ---
nasa_files = glob.glob(os.path.join('data', 'raw', 'nasapower', '*.csv'))
for src in nasa_files:
    fname = os.path.basename(src)
    dst = os.path.join(DIRS['raw_nasa'], fname)
    if not os.path.exists(dst):
        shutil.copy2(src, dst)
        print(f"  ✅ NASA copiado: {fname}")

print("\n✅ Todos los archivos fuente (MIDAGRI, INDECI, Noticias, NASA) sincronizados.")


  ℹ️  MIDAGRI ya existe en destino.
  ℹ️  INDECI ya existe: resumen_emergencias_2003_2024.xlsx
  ℹ️  INDECI ya existe: resumen_peligros_2003_2024.xlsx
  ℹ️  INDECI ya existe: piura_emergencias.xlsx
  ℹ️  INDECI ya existe: piura_peligros.xlsx
  ℹ️  INDECI DBF ya existe: E_2021/
  ℹ️  INDECI DBF ya existe: E_2022/
  ℹ️  INDECI DBF ya existe: E_2023/
  ℹ️  AGRARIA ya existe: agro_news_2021.csv
  ℹ️  AGRARIA ya existe: agro_news_2022.csv
  ℹ️  AGRARIA ya existe: agro_news_2023.csv
  ℹ️  AGRARIA ya existe: agro_news_2024.csv
  ℹ️  AGRARIA ya existe: agro_news_2025.csv

✅ Todos los archivos fuente (MIDAGRI, INDECI, Noticias, NASA) sincronizados.


In [4]:
# ==========================================================================
# 1.4 Definición de Constantes del Proyecto
# ==========================================================================
# Estas constantes se reutilizarán en todos los notebooks.

# --- Rango temporal del estudio ---
ANHO_INICIO = 2021
ANHO_FIN = 2025
MES_FIN = 8  # Agosto 2025 (último mes con data completa)

# --- Cultivo objetivo ---
CULTIVO_TARGET = 'LIMON'

# --- Conexión PostgreSQL ---
PG_USER = 'postgres'
PG_PASS = 'postgres'
PG_HOST = 'localhost'
PG_PORT = '5432'
PG_DB   = 'limon_analytics_db'
PG_URI  = f'postgresql://{PG_USER}:{PG_PASS}@{PG_HOST}:{PG_PORT}/{PG_DB}'

# --- Peligros hidrometeorológicos válidos (INDECI) ---
PELIGROS_VALIDOS = [
    'LLUVIAS INTENSAS', 'INUNDACION', 'HUAYCO', 'SEQUIA',
    'HELADAS', 'FRIAJE', 'GRANIZADA', 'NEVADA', 'VIENTOS FUERTES',
    'DESLIZAMIENTO', 'EROSION'
]

# Guardar constantes para reutilización entre notebooks
import json
config = {
    'ANHO_INICIO': ANHO_INICIO, 'ANHO_FIN': ANHO_FIN, 'MES_FIN': MES_FIN,
    'CULTIVO_TARGET': CULTIVO_TARGET, 'PG_URI': PG_URI,
    'PELIGROS_VALIDOS': PELIGROS_VALIDOS,
    'DIRS': DIRS,
}
config_path = os.path.join(DIRS['interim'], 'pipeline_config.json')
with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print(f"✅ Configuración guardada en: {config_path}")
print(f"\n[ACTIVIDAD 1] COMPLETADA.")
print(f"  Descripción: Configuración del entorno, estructura de carpetas y constantes.")
print(f"  Resultado: {len(DIRS)} directorios creados, archivos fuente copiados.")
print(f"  Archivo generado: {config_path}")


✅ Configuración guardada en: data\02_interim\pipeline_config.json

[ACTIVIDAD 1] COMPLETADA.
  Descripción: Configuración del entorno, estructura de carpetas y constantes.
  Resultado: 12 directorios creados, archivos fuente copiados.
  Archivo generado: data\02_interim\pipeline_config.json
